Roadmap item 3: a real, runnable "Semantic RAG" baseline — semantic
similarity only, no hierarchy — to compare honestly against MSGR-RAG.

Updated to point at:
- the corrected graph_entity.gpickle (post K=2 domain-graph fix; G_E itself
was never touched by that fix, but re-pointing for consistency with the
current pipeline paths)
- test_facts_large_v2.csv (981 facts, 4 categories) instead of the old
29-fact hand-built set

Approach: for each entity, build a text corpus from its known real facts
("Aspirin treats Pain", "Aspirin causes Bleeding", ...). For a candidate
fact, compute TF-IDF cosine similarity between the candidate fact's text
and the subject entity's corpus of known real facts. High similarity to
known real facts => predicted VERIFIED; low => predicted HALLUCINATION.

In [1]:
import pandas as pd
import pickle
from pathlib import Path
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

INTERMEDIATE_DIR = Path("../Datasets/intermediate")

GRAPH_ENTITY_PATH = INTERMEDIATE_DIR / "graph_entity.gpickle"
TEST_FACTS_PATH = INTERMEDIATE_DIR / "test_facts_large_v2.csv"
PREDICTIONS_OUTPUT_PATH = INTERMEDIATE_DIR / "semantic_rag_predictions.csv"
REPORT_OUTPUT_PATH = INTERMEDIATE_DIR / "semantic_rag_report.json"

SIMILARITY_THRESHOLD = 0.35  # starting point; can be swept the same way as MSGR-RAG's thresholds


In [2]:
def fact_to_text(subject, predicate, obj):
    return f"{subject} {predicate.replace('_', ' ')} {obj}"


In [3]:
def build_entity_corpora(G_E):
    """For each entity, collect text of all its real known outgoing facts."""
    corpora = {}
    for u, v, d in G_E.edges(data=True):
        text = fact_to_text(u, d["predicate"], v)
        corpora.setdefault(u, []).append(text)
    return corpora

In [4]:
def semantic_rag_predict(test_facts_df, G_E):
    corpora = build_entity_corpora(G_E)
    predictions = []

    for _, row in test_facts_df.iterrows():
        s, p, o = row["subject"], row["predicate"], row["object"]
        candidate_text = fact_to_text(s, p, o)
        known_facts = corpora.get(s, [])

        if not known_facts:
            predictions.append({"decision": "HALLUCINATION", "similarity": 0.0})
            continue

        vectorizer = TfidfVectorizer().fit(known_facts + [candidate_text])
        vectors = vectorizer.transform(known_facts + [candidate_text])
        candidate_vec = vectors[-1]
        known_vecs = vectors[:-1]

        sims = cosine_similarity(candidate_vec, known_vecs)[0]
        max_sim = float(sims.max())

        decision = "VERIFIED" if max_sim >= SIMILARITY_THRESHOLD else "HALLUCINATION"
        predictions.append({"decision": decision, "similarity": max_sim})

    return pd.DataFrame(predictions)

In [5]:
def compute_metrics(df):
    TP = len(df[(df.ground_truth == "HALLUCINATION") & (df.decision == "HALLUCINATION")])
    TN = len(df[(df.ground_truth == "VERIFIED") & (df.decision == "VERIFIED")])
    FP = len(df[(df.ground_truth == "VERIFIED") & (df.decision == "HALLUCINATION")])
    FN = len(df[(df.ground_truth == "HALLUCINATION") & (df.decision == "VERIFIED")])

    total = len(df)
    accuracy = (TP + TN) / total if total else 0
    precision = TP / (TP + FP) if (TP + FP) else 0
    recall = TP / (TP + FN) if (TP + FN) else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0

    return {"TP": TP, "TN": TN, "FP": FP, "FN": FN,
            "accuracy": accuracy, "precision": precision, "recall": recall, "f1": f1}


In [6]:
def main():
    print("Loading data...")
    with open(GRAPH_ENTITY_PATH, "rb") as f:
        G_E = pickle.load(f)
    test_facts_df = pd.read_csv(TEST_FACTS_PATH)
    print(f"Loaded {len(test_facts_df)} test facts")
    print(test_facts_df.groupby(["category", "ground_truth"]).size())

    print(f"\nRunning Semantic RAG baseline (threshold={SIMILARITY_THRESHOLD})...")
    preds = semantic_rag_predict(test_facts_df, G_E)

    merged = pd.concat([test_facts_df.reset_index(drop=True), preds], axis=1)
    merged.to_csv(PREDICTIONS_OUTPUT_PATH, index=False)

    overall = compute_metrics(merged)
    print(f"\n--- Overall Semantic RAG performance ---")
    print(f"Accuracy:  {overall['accuracy']*100:.1f}%")
    print(f"Precision: {overall['precision']*100:.1f}%")
    print(f"Recall:    {overall['recall']*100:.1f}%")
    print(f"F1:        {overall['f1']*100:.1f}%")
    print(f"TP={overall['TP']} TN={overall['TN']} FP={overall['FP']} FN={overall['FN']}")

    print(f"\n--- Per-category accuracy (for direct comparison with MSGR-RAG) ---")
    category_breakdown = {}
    for cat in merged["category"].unique():
        cat_df = merged[merged["category"] == cat]
        correct = (cat_df["ground_truth"] == cat_df["decision"]).sum()
        total = len(cat_df)
        acc = correct / total if total else 0
        category_breakdown[cat] = {"n": int(total), "correct": int(correct), "accuracy": float(acc)}
        print(f"  {cat:30s}  n={total:4d}  correct={correct:4d}  accuracy={acc*100:5.1f}%")

    report = {
        "generation_date": pd.Timestamp.now().strftime("%Y-%m-%d %H:%M:%S"),
        "similarity_threshold": SIMILARITY_THRESHOLD,
        "test_set_size": len(merged),
        "overall_performance": overall,
        "category_breakdown": category_breakdown,
    }

    import json
    with open(REPORT_OUTPUT_PATH, "w") as f:
        json.dump(report, f, indent=2)

    print(f"\nSaved predictions: {PREDICTIONS_OUTPUT_PATH}")
    print(f"Saved report: {REPORT_OUTPUT_PATH}")

In [7]:
main()

Loading data...
Loaded 981 test facts
category                  ground_truth 
domain_violation          HALLUCINATION     84
nonexistent_relationship  HALLUCINATION    297
real_edge                 VERIFIED         400
wrong_predicate           HALLUCINATION    200
dtype: int64

Running Semantic RAG baseline (threshold=0.35)...

--- Overall Semantic RAG performance ---
Accuracy:  67.4%
Precision: 100.0%
Recall:    44.9%
F1:        62.0%
TP=261 TN=400 FP=0 FN=320

--- Per-category accuracy (for direct comparison with MSGR-RAG) ---
  nonexistent_relationship        n= 297  correct= 219  accuracy= 73.7%
  real_edge                       n= 400  correct= 400  accuracy=100.0%
  wrong_predicate                 n= 200  correct=   0  accuracy=  0.0%
  domain_violation                n=  84  correct=  42  accuracy= 50.0%

Saved predictions: ../Datasets/intermediate/semantic_rag_predictions.csv
Saved report: ../Datasets/intermediate/semantic_rag_report.json


In [1]:
# ============================================================================
# Simple RAG baseline (no MSGR-RAG verification): retrieval + generation only.
# Tested on the SAME 974 natural questions as Component 6, for a fair,
# like-for-like "plain RAG vs. RAG+MSGR-RAG" comparison.
# ============================================================================

import pandas as pd
import pickle
import requests
import json
from pathlib import Path
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

INTERMEDIATE_DIR = Path("../Datasets/intermediate")
OLLAMA_URL = "http://localhost:11434/api/generate"
OLLAMA_MODEL = "gemma3:latest"

with open(INTERMEDIATE_DIR / "graph_entity.gpickle", "rb") as f:
    G_E = pickle.load(f)

natural_questions_df = pd.read_csv(INTERMEDIATE_DIR / "component6_natural_questions.csv")
print(f"Loaded {len(natural_questions_df)} natural questions")


def fact_to_text(subject, predicate, obj):
    return f"{subject} {predicate.replace('_', ' ')} {obj}"


def build_entity_corpora(G_E):
    corpora = {}
    for u, v, d in G_E.edges(data=True):
        text = fact_to_text(u, d["predicate"], v)
        corpora.setdefault(u, []).append(text)
    return corpora


corpora = build_entity_corpora(G_E)


def retrieve_context(subject, top_k=5):
    """
    Retrieval step: find the top-k most similar known facts about the
    subject entity, using the same TF-IDF approach as the similarity
    baseline, for methodological consistency.
    """
    known_facts = corpora.get(subject, [])
    if not known_facts:
        return []
    return known_facts[:top_k]  # already the entity's known real facts


RAG_ANSWER_PROMPT_TEMPLATE = """You are a medical assistant answering a patient's question using
retrieved facts from a drug database. Only use the retrieved facts below - do not use
outside knowledge. If the retrieved facts do not support a clear answer, say so.

Retrieved facts about {subject}:
{context}

Patient question: {question}

Give a short, direct answer (2-3 sentences), then on a new final line write
exactly one of: ANSWER: VERIFIED or ANSWER: HALLUCINATION
- Write VERIFIED if the retrieved facts support the claim in the question.
- Write HALLUCINATION if the retrieved facts contradict or do not support the claim.
"""


def run_simple_rag_v2(question, subject, model=OLLAMA_MODEL):
    """
    Plain RAG: retrieve context, generate an answer, extract the LLM's own
    self-reported decision. No external verification of any kind - this is
    the baseline MSGR-RAG's Component 6 is meant to improve upon.
    """
    context_facts = retrieve_context(subject)
    context_text = "\n".join(f"- {f}" for f in context_facts) if context_facts else "(no retrieved facts found)"

    prompt = RAG_ANSWER_PROMPT_TEMPLATE.format(subject=subject, context=context_text, question=question)

    try:
        response = requests.post(
            OLLAMA_URL,
            json={"model": model, "prompt": prompt, "stream": False, "options": {"temperature": 0.0}},
            timeout=60
        )
        response.raise_for_status()
        answer_text = response.json()["response"].strip()

        if "ANSWER: VERIFIED" in answer_text:
            decision = "VERIFIED"
        elif "ANSWER: HALLUCINATION" in answer_text:
            decision = "HALLUCINATION"
        else:
            decision = "UNPARSEABLE"

        return {
            'decision': decision,
            'full_answer': answer_text,
            'context_used': context_facts,
            'error': None,
        }
    except Exception as e:
        return {'decision': 'ERROR', 'full_answer': None, 'context_used': [], 'error': str(e)}


# ---- Quick test on 3 questions before the full 974-question run ----
test_sample = natural_questions_df.sample(3, random_state=5)
for _, row in test_sample.iterrows():
    result = run_simple_rag(row['generated_question'], row['subject'])
    print(f"\nQ: {row['generated_question']}")
    print(f"Ground truth: {row['ground_truth']}")
    print(f"Simple RAG decision: {result['decision']}")
    print(f"Answer: {result['full_answer']}")

Loaded 974 natural questions

Q: My doctor prescribed me acetylsalicylic acid for my arthritis, but I've heard it can cause fungal infections – is there a connection?
Ground truth: HALLUCINATION
Simple RAG decision: HALLUCINATION
Answer: I understand your concern about acetylsalicylic acid and fungal infections. However, I’ve been unable to find any retrieved facts regarding a connection between this medication and fungal infections. Therefore, I cannot confirm or deny this potential link.

ANSWER: HALLUCINATION

Q: I was prescribed Everolimus with Dimenhydrinate, will that combination be safe for me?
Ground truth: HALLUCINATION
Simple RAG decision: HALLUCINATION
Answer: Based on the information available, the combination of Everolimus with Dimenhydrinate may not be safe due to potential interactions with other medications you may be taking. Specifically, Everolimus interacts with Teriflunomide, Warfarin, Telotristat ethyl, Midecamycin, and Lobeglitazone.

ANSWER: HALLUCINATION

Q: My 

In [7]:
# ============================================================================
# Full run (v2, corrected retrieval): Simple RAG on all 974 natural questions
# ============================================================================

import time

CHECKPOINT_PATH_RAG_V2 = INTERMEDIATE_DIR / "simple_rag_v2_results_checkpoint.csv"
FINAL_OUTPUT_PATH_RAG_V2 = INTERMEDIATE_DIR / "simple_rag_v2_results.csv"
CHECKPOINT_EVERY = 100

rag_v2_results = []
start = time.time()

for i, row in natural_questions_df.iterrows():
    result = run_simple_rag_v2(row['generated_question'], row['subject'])
    rag_v2_results.append({
        'subject': row['subject'],
        'predicate': row['predicate'],
        'object': row['object'],
        'category': row['category'],
        'ground_truth': row['ground_truth'],
        'generated_question': row['generated_question'],
        'simple_rag_decision': result['decision'],
        'full_answer': result['full_answer'],
        'error': result['error'],
    })

    if (i + 1) % CHECKPOINT_EVERY == 0:
        elapsed = time.time() - start
        remaining = (elapsed / (i + 1)) * (len(natural_questions_df) - (i + 1))
        print(f"  {i+1}/{len(natural_questions_df)} done "
              f"({elapsed:.0f}s elapsed, ~{remaining:.0f}s remaining)")
        pd.DataFrame(rag_v2_results).to_csv(CHECKPOINT_PATH_RAG_V2, index=False)

rag_v2_df = pd.DataFrame(rag_v2_results)
rag_v2_df.to_csv(FINAL_OUTPUT_PATH_RAG_V2, index=False)

total_time = time.time() - start
print(f"\nDone. Total time: {total_time:.0f}s ({total_time/60:.1f} min)")
print(f"\nDecision distribution:")
print(rag_v2_df['simple_rag_decision'].value_counts())
print(f"\nErrors: {rag_v2_df['error'].notna().sum()}")

scoreable = rag_v2_df[rag_v2_df['simple_rag_decision'].isin(['VERIFIED', 'HALLUCINATION'])]
correct = (scoreable['simple_rag_decision'] == scoreable['ground_truth']).sum()
print(f"\nScoreable: {len(scoreable)}/{len(rag_v2_df)}")
print(f"Accuracy: {correct}/{len(scoreable)} = {correct/len(scoreable)*100:.1f}%")

print(f"\nPer-category accuracy:")
for cat in scoreable['category'].unique():
    cat_df = scoreable[scoreable['category'] == cat]
    cat_correct = (cat_df['simple_rag_decision'] == cat_df['ground_truth']).sum()
    print(f"  {cat:28s} n={len(cat_df):4d}  accuracy={cat_correct/len(cat_df)*100:5.1f}%")

print(f"\nSaved: {FINAL_OUTPUT_PATH_RAG_V2}")

KeyboardInterrupt: 

In [6]:
import pandas as pd

# Compare the two result files directly - same facts, do they get the same decision?
run1 = pd.read_csv("../Datasets/intermediate/plain_rag_sensitivity.csv")
# If you have a second output file from the new run, load it here with its actual filename
# run2 = pd.read_csv("../Datasets/intermediate/<whatever the new run saved as>.csv")

print(f"Run 1 shape: {run1.shape}")
print(run1['simple_rag_decision'].value_counts())

Run 1 shape: (974, 13)
simple_rag_decision
HALLUCINATION    488
VERIFIED         486
Name: count, dtype: int64


In [8]:
# ============================================================================
# Determinism check: run the SAME question through Ollama 5 times,
# temperature=0.0, and see if we get identical output every time.
# ============================================================================

test_question = "Will Aspirin cause stomach bleeding?"
test_subject = "Acetylsalicylic acid"

print("Running the same question 5 times to check determinism...\n")
for i in range(5):
    result = run_simple_rag_v2(test_question, test_subject)
    print(f"Run {i+1}: decision={result['decision']}")
    print(f"  Answer: {result['full_answer'][:150]}...")
    print()

Running the same question 5 times to check determinism...

Run 1: decision=VERIFIED
  Answer: Yes, Acetylsalicylic acid, commonly known as Aspirin, causes bleeding. The drug database indicates that it is associated with this side effect.

ANSWE...

Run 2: decision=VERIFIED
  Answer: Yes, Acetylsalicylic acid, commonly known as Aspirin, causes bleeding. The drug database indicates that it is associated with this side effect.

ANSWE...

Run 3: decision=VERIFIED
  Answer: Yes, Acetylsalicylic acid, commonly known as Aspirin, causes bleeding. The drug database indicates that it is associated with this side effect.

ANSWE...

Run 4: decision=VERIFIED
  Answer: Yes, Acetylsalicylic acid, commonly known as Aspirin, causes bleeding. The drug database indicates that it is associated with this side effect.

ANSWE...

Run 5: decision=VERIFIED
  Answer: Yes, Acetylsalicylic acid, commonly known as Aspirin, causes bleeding. The drug database indicates that it is associated with this side effect.

A